In [30]:
import pandas as pd
import glob

In [31]:
files = glob.glob('../data/*.csv')
df = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)

print(df.shape)
print(df['Team'].value_counts())
df.head()

(2227, 13)
Team
Crewmate    1761
Imposter     466
Name: count, dtype: int64


,Game Completed Date,Team,Outcome,Task Completed,All Tasks Completed,Murdered,Imposter Kills,Game Length,Ejected,Sabotages Fixed,Time to complete all tasks,Rank Change,Region/Game Code
0,12/13/2020 at 1:26:56 am EST,Crewmate,Win,3,No,Yes,-,07m 04s,No,2.0,-,++,NA / WYMSBF
1,12/13/2020 at 1:17:42 am EST,Crewmate,Loss,7,Yes,No,-,16m 21s,No,1.0,09m 48s,--,NA / WYMSBF
2,12/13/2020 at 12:57:47 am EST,Crewmate,Win,3,No,No,-,11m 33s,No,0.0,-,++,NA / WYMSBF
3,12/13/2020 at 12:41:55 am EST,Imposter,Win,-,-,-,2,08m 05s,No,NaN,-,+++,Europe / QIRTNF
4,12/13/2020 at 12:30:37 am EST,Crewmate,Loss,4,No,No,-,05m 10s,No,0.0,-,---,Europe / QIRTNF


In [32]:
#convert game length strings to int seconds
def parse_time(s):
    if s == "-":
        return None
    parts = s.split("m ")
    minutes = int(parts[0])
    seconds = int(parts[1][:-1])
    total = minutes * 60 + seconds
    return total

In [33]:
df['Game Length (s)'] = df['Game Length'].apply(parse_time)

df['Task Time (s)'] = df['Time to complete all tasks'].apply(parse_time)
df['Task Time (s)'] = df['Task Time (s)'].fillna(0)

#changing yes/no, win/lose values to 1/0 for later analysis
df['All Tasks Completed'] = df['All Tasks Completed'].map({'Yes':1, 'No': 0, '-': 0})

df['Murdered'] = df['Murdered'].map({'Yes':1, 'No': 0, '-': None})
df['Murdered'] = df['Murdered'].fillna(0)

df['Ejected'] = df['Ejected'].map({'Yes':1, 'No': 0, '-': None})
df['Outcome'] = df['Outcome'].map({'Win':1, 'Loss': 0})

#fill in Nan - for imposter kills, task completed, sabatagoes fixed
df['Imposter Kills'] = pd.to_numeric(df['Imposter Kills'], errors='coerce')
df['Imposter Kills'] = df['Imposter Kills'].fillna(0)

df['Task Completed'] = pd.to_numeric(df['Task Completed'], errors='coerce')
df['Task Completed'] = df['Task Completed'].fillna(0)

df['Sabotages Fixed'] = pd.to_numeric(df['Sabotages Fixed'], errors='coerce')
df['Sabotages Fixed'] = df['Sabotages Fixed'].fillna(0)

#create is_imposter column as 1/0 column from team column
df['is_imposter'] = df['Team'].map({'Imposter':1, 'Crewmate': 0})

In [34]:
#checking null values
df.isnull().sum()

Game Completed Date             0
Team                            0
Outcome                         0
Task Completed                  0
All Tasks Completed             0
Murdered                        0
Imposter Kills                  0
Game Length                     0
Ejected                         0
Sabotages Fixed                 0
Time to complete all tasks      0
Rank Change                   399
Region/Game Code                0
Game Length (s)                 0
Task Time (s)                   0
is_imposter                     0
dtype: int64

In [35]:
#keeping needed columns
features = ['Outcome', 'Task Completed', 'All Tasks Completed', 'Murdered', 'Imposter Kills', 'Ejected', 'Sabotages Fixed', 'Game Length (s)', 'Task Time (s)', 'is_imposter']
df_clean = df[features]

print(df_clean.shape)
df_clean.head()

(2227, 10)


,Outcome,Task Completed,All Tasks Completed,Murdered,Imposter Kills,Ejected,Sabotages Fixed,Game Length (s),Task Time (s),is_imposter
0,1,3.0,0,1.0,0.0,0.0,2.0,424,0.0,0
1,0,7.0,1,0.0,0.0,0.0,1.0,981,588.0,0
2,1,3.0,0,0.0,0.0,0.0,0.0,693,0.0,0
3,1,0.0,0,0.0,2.0,0.0,0.0,485,0.0,1
4,0,4.0,0,0.0,0.0,0.0,0.0,310,0.0,0


In [36]:
df_clean.to_csv('../data/cleaned_data.csv', index=False)